# Global Topic Renaming (Part 2)

The per-cluster naming in `get_topic_names.ipynb` works locally: each topic is
named in isolation. This can produce **duplicate or ambiguous names** when two
clusters cover related sub-themes (e.g. both named "History of Synthetic Biology").

This notebook fixes that by giving the LLM a **global view** of all topics at
once. We use **OpenAI function calling** so the model returns a structured array
of `(topic_id, name)` pairs — one per cluster — guaranteeing distinct,
publication-ready names.

In [ ]:
import os
import json
import yaml
import pandas as pd
from openai import OpenAI

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR = os.path.join('..', 'assets')
MODELS_DIR = os.path.join(ASSETS_DIR, 'topic_models')

MODEL = 'gpt-4.1-nano'

with open('prompts.yaml') as f:
    prompts = yaml.safe_load(f)

THEME = prompts['theme']
THEME_DESCRIPTION = prompts['theme_description']

# ── OpenAI client ─────────────────────────────────────────────────────────────
api_key = open('openai.key').read().strip()
client = OpenAI(api_key=api_key)

## 1. Load part-1 results

In [ ]:
teams_names  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'),  sep='\t')
papers_names = pd.read_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t')

print(f'Teams:  {len(teams_names)} topics')
print(f'Papers: {len(papers_names)} topics')
teams_names[['topic', 'name', 'description']].head()

## 2. Function-calling tool definition and renaming logic

In [ ]:
def build_rename_tool(n_topics: int) -> dict:
    """Build the OpenAI function-calling tool schema for renaming n topics."""
    return {
        'type': 'function',
        'function': {
            'name': 'rename_topics',
            'description': (
                f'Assign a unique, concise, publication-ready name to each of '
                f'the {n_topics} topic clusters. Every name must be distinct.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'topics': {
                        'type': 'array',
                        'description': f'Exactly {n_topics} renamed topics.',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'topic_id': {
                                    'type': 'integer',
                                    'description': 'The original topic ID.',
                                },
                                'name': {
                                    'type': 'string',
                                    'description': (
                                        'A short, unique name for this topic '
                                        '(3-7 words). Must be distinct from '
                                        'every other topic name.'
                                    ),
                                },
                            },
                            'required': ['topic_id', 'name'],
                        },
                    },
                },
                'required': ['topics'],
            },
        },
    }


def build_user_message(names_df: pd.DataFrame) -> str:
    """Format all topics into a single user message for the LLM."""
    lines = []
    for _, row in names_df.iterrows():
        lines.append(
            f"Topic {int(row['topic'])}:\n"
            f"  Current name: {row['name']}\n"
            f"  Description: {row['description']}\n"
        )
    return '\n'.join(lines)


SYSTEM_PROMPT = (
    f'You are a research consultant with expertise on {THEME}, meaning that '
    f'you know about {THEME_DESCRIPTION}\n\n'
    f'You will receive a list of topic clusters, each with an ID, a current '
    f'name, and a description. Some current names may be duplicated or '
    f'ambiguous. Your task is to assign a NEW, UNIQUE, short name to each '
    f'topic that:\n'
    f'1. Clearly distinguishes it from every other topic in the list.\n'
    f'2. Faithfully reflects the description.\n'
    f'3. Is concise (3-7 words) and suitable for a figure legend in an '
    f'academic publication.\n'
    f'4. No two names should overlap or be paraphrases of each other.\n\n'
    f'Return ALL topics, even if the current name is already good — confirm '
    f'or improve each one.'
)

In [ ]:
def rename_topics_global(names_df: pd.DataFrame) -> pd.DataFrame:
    """Call OpenAI with function calling to get globally distinct topic names.

    Returns a copy of names_df with a new column `global_name`.
    """
    n = len(names_df)
    tool = build_rename_tool(n)
    user_msg = build_user_message(names_df)

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        tools=[tool],
        tool_choice={'type': 'function', 'function': {'name': 'rename_topics'}},
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
    )

    # Parse the function call arguments
    call = response.choices[0].message.tool_calls[0]
    args = json.loads(call.function.arguments)
    renamed = {item['topic_id']: item['name'] for item in args['topics']}

    result = names_df.copy()
    result['global_name'] = result['topic'].map(renamed)

    # Verify all topics got a name
    missing = result['global_name'].isna().sum()
    if missing > 0:
        print(f'WARNING: {missing} topic(s) did not receive a global name')

    # Verify uniqueness
    dupes = result['global_name'].duplicated().sum()
    if dupes > 0:
        print(f'WARNING: {dupes} duplicate global name(s) found')

    return result

## 3. Rename: iGEM Teams topics

In [ ]:
teams_renamed = rename_topics_global(teams_names)
teams_renamed[['topic', 'name', 'global_name', 'description']]

## 4. Rename: SynBio Papers topics

In [ ]:
papers_renamed = rename_topics_global(papers_names)
papers_renamed[['topic', 'name', 'global_name', 'description']]

## 5. Save final results

In [ ]:
# Overwrite the part-1 files with the global_name column added
teams_renamed.to_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'),  sep='\t', index=False)
papers_renamed.to_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t', index=False)

print(f'Saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')